# Fine-tune Llama 3.1 8B with Unsloth

In [1]:
import unsloth
from torch import __version__
from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"

import torch
from trl import SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments, TextStreamer
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel, is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Load model for PEFT

In [2]:
# Load model
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Prepare model for PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth"
)
print(model.print_trainable_parameters())

==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GH200 120GB. Num GPUs = 1. Max memory: 95.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
None


## 2. Prepare data and tokenizer

In [3]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

# Load the recipes dataset
from datasets import Dataset, load_dataset

dataset_subset = load_dataset("Hieu-Pham/kaggle_food_recipes", split="train")

messages_list = []
skipped_long = 0
skipped_error = 0
skipped_ingredients = 0

for row in dataset_subset:
    try:
        if not row["Instructions"] or len(row["Instructions"].strip()) < 50:
            continue

        user_message = {
            "role": "user",
            "content": f"Please write me a recipe for {row['Title']}."
        }

        ingredients = eval(row["Cleaned_Ingredients"])
        # if len(ingredients) > 20:
        #     skipped_ingredients += 1
        #     continue

        assistant_content = "Ingredients:\n"
        assistant_content += "\n".join(ingredients)
        assistant_content += "\n\nInstructions:\n"
        assistant_content += row["Instructions"].strip()

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        # Compute token length to filter out long examples
        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False
        )
        input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
        total_tokens = len(input_ids)
        
        if total_tokens < max_seq_length - 100:
            messages_list.append({"conversations": [user_message, assistant_message]})
        else:
            skipped_long += 1
    except Exception as e:
        skipped_error += 1

print(f"Subset loaded: {len(dataset_subset)} examples")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")
print(f"Skipped (ingredients): {skipped_ingredients}")

dataset = Dataset.from_list(messages_list)

def apply_template(examples):
    messages = examples["conversations"]
    text = [tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=False) for message in messages]
    return {"text": text}

dataset = dataset.map(apply_template, batched=True)

print(f"Dataset created with {len(dataset)} examples")
print(f"\nExample recipe:\n{dataset[0]['conversations'][1]['content'][:500]}...")

Unsloth: Will map <|im_end|> to EOS = <|end_of_text|>.


Subset loaded: 13501 examples
Skipped (too long): 4
Skipped (errors): 0
Skipped (ingredients): 0


Map:   0%|          | 0/13486 [00:00<?, ? examples/s]

Dataset created with 13486 examples

Example recipe:
Ingredients:
1 (3½–4-lb.) whole chicken
2¾ tsp. kosher salt, divided, plus more
2 small acorn squash (about 3 lb. total)
2 Tbsp. finely chopped sage
1 Tbsp. finely chopped rosemary
6 Tbsp. unsalted butter, melted, plus 3 Tbsp. room temperature
¼ tsp. ground allspice
Pinch of crushed red pepper flakes
Freshly ground black pepper
⅓ loaf good-quality sturdy white bread, torn into 1" pieces (about 2½ cups)
2 medium apples (such as Gala or Pink Lady; about 14 oz. total), cored, cut into 1" pieces
2 T...


## 3. Training

In [4]:
from trl import SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        max_seq_length=max_seq_length,
        packing=False,  # Disabled due to Unsloth 2025.12.4 bug with packing
        learning_rate=3e-5,
        lr_scheduler_type="linear",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        seed=0,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/13486 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,486 | Num Epochs = 1 | Total steps = 422
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.531800
2,1.612700
3,1.607500
4,1.646200
5,1.707800
6,1.551800
7,1.492500
8,1.560000
9,1.395100
10,1.427300


train/epoch,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,█▇▅▇▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▂▁▁▂▁▂▂▂▁▂▁▂▁▁
train/learning_rate,███▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▁▁▁▁
train/loss,█▇▄▄▂▄▄▄▄▃▂▃▃▂▂▂▂▃▃▁▂▁▂▃▃▂▁▂▃▂▁▃▂▂▃▂▂▁▂▁
total_flos,4.348480120053596e+17
train/epoch,1
train/global_step,422
train/grad_norm,0.87769
train/learning_rate,0.0
train/loss,1.0553


TrainOutput(global_step=422, training_loss=1.0527090414722948, metrics={'train_runtime': 1104.5818, 'train_samples_per_second': 12.209, 'train_steps_per_second': 0.382, 'total_flos': 4.348480120053596e+17, 'train_loss': 1.0527090414722948, 'epoch': 1.0})

## 4. Inference

In [5]:
# Load model for inference
model = FastLanguageModel.for_inference(model)

messages = [
    {"from": "human", "value": "Is 9.11 larger than 9.9?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128, use_cache=True)

UndefinedError: 'dict object' has no attribute 'content'

## 5. Save trained model

In [ ]:
model.save_pretrained_merged("model", tokenizer, save_method="merged_16bit")
model.push_to_hub_merged("mlabonne/FineLlama-3.1-8B", tokenizer, save_method="merged_16bit")

In [ ]:
model.save_pretrained_gguf("model", tokenizer, "q8_0")
quant_methods = ["q2_k", "q3_k_m", "q4_k_m", "q5_k_m", "q6_k", "q8_0"]
for quant in quant_methods:
    model.push_to_hub_gguf("mlabonne/FineLlama-3.1-8B-GGUF", tokenizer, quant)